# Business Entity Resolution — hybrid multilingual baseline

Integrity check → normalization + optional transliteration → TF-IDF + BM25 → global top-K candidates → fixed pair features → LightGBM (primary) and SGD (reference) → calibration-only threshold → untouched holdout → saved model → fresh test indexes.

CPU works; GPU is optional only for LightGBM. Default: **20,000 sampled S1 entities**, full target catalog and a **global K=20** candidate cap. K=50 is diagnostic only. Begin with 2,000 S1 for runtime profiling if needed. No official-data accuracy/runtime is claimed.

Put exactly one copy of the four training files (TSV or ZIP) in `MyDrive/AmazonML2026/input/train`. Later put three test source files under `input/test`.


In [ ]:
from pathlib import Path
import subprocess, sys, json, shutil, hashlib
REPO = Path('/content/Amazon-ML-Challange-2026')
URL = 'https://github.com/alisalmann7386-crypto/Amazon-ML-Challange-2026.git'
if not REPO.exists():
    subprocess.run(['git','clone',URL,str(REPO)],check=True)
else:
    dirty = subprocess.check_output(['git','status','--porcelain'],cwd=REPO,text=True).strip()
    if dirty:
        raise RuntimeError('Existing checkout has local changes; save them before updating. Nothing was overwritten.')
    subprocess.run(['git','pull','--ff-only'],cwd=REPO,check=True)
print('Source commit:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())
PY = sys.executable  # Colab images may not include the python3-venv system package.
subprocess.run([PY,'-m','pip','install','-r',str(REPO/'requirements.txt')],check=True)
def run(*args):
    subprocess.run([PY,'-u',*map(str,args)],cwd=REPO,check=True)
run('-m','unittest','discover','-s','tests','-v')


## Configure and restore checkpoints
Set `RUN_TAG` to a new value when changing input datasets. A parameter-derived key separates different configurations. Data/config mismatches cause an explicit error.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/AmazonML2026')
SAMPLE_SIZE = 20000
EDA_MAX_ROWS = 10000  # 0 = full streaming EDA; 10000 = labeled prefix sample.
LIGHTGBM_DEVICE = 'cpu'  # Optional 'gpu'; unsupported backend falls back to CPU.
RUN_TAG = 'integrity_v2_k20'
RUN_TEST = False
TEAM = 'YOUR_TEAM'
cfg = json.loads((REPO/'configs/baseline.json').read_text())
cfg['sample_size'] = SAMPLE_SIZE
cfg['blocking']['final_k'] = 20
cfg['model']['device_type'] = LIGHTGBM_DEVICE
# Optional: cfg['transliteration']['enabled'] = False
# Optional: cfg['normalization']['ambiguous_st_to_street'] = True
key = RUN_TAG + '_' + hashlib.sha256(json.dumps(cfg,sort_keys=True).encode()).hexdigest()[:10]
LOCAL = Path('/content/er_hybrid')/key
LOCAL.mkdir(parents=True,exist_ok=True)
ART = LOCAL/'artifacts'
SAVED = DRIVE_ROOT/'artifacts'/key
if SAVED.exists() and not ART.exists():
    shutil.copytree(SAVED,ART)
ART.mkdir(exist_ok=True)
CONFIG = LOCAL/'config.json'
CONFIG.write_text(json.dumps(cfg,indent=2))
DATA = LOCAL/'dataset'
INDEX = ART/'index_train'
WORK = ART/'run'
FINAL = ART/'final_model'
TEST_INDEX = ART/'index_test'
OUTPUT = ART/'output_hybrid'
EXACT_OUTPUT = ART/'output_exact'
def checkpoint():
    shutil.copytree(ART,SAVED,dirs_exist_ok=True,ignore=shutil.ignore_patterns('*.tmp','*.tmp.npz','*.building*','train_X.npy','train_y.npy'))
def phase(*commands):
    try:
        for command in commands: run(*command)
    finally:
        checkpoint()
print('Free local disk GB:',round(shutil.disk_usage('/content').free/1e9,2))
print('Local artifacts:',ART)
print('Drive checkpoint:',SAVED)


## Prepare files, prove integrity, then run EDA
The copy manifest proves byte/line preservation. The integrity scan records recoverable Source-3 repairs and stops on duplicate IDs or ground-truth IDs absent from the completed files. EDA runs only after this passes.

In [ ]:
run('src/prepare_data.py','--input',DRIVE_ROOT/'input/train','--output',DATA/'train','--split','train')
phase(['src/data_integrity.py','--data',DATA/'train','--split','train','--output',ART/'data_integrity.json'])
print(json.dumps(json.loads((ART/'data_integrity.json').read_text()),indent=2,ensure_ascii=False))
phase(['src/eda.py','--data',DATA/'train','--config',CONFIG,'--max-rows',EDA_MAX_ROWS,'--output',ART/'eda'])
print(json.dumps(json.loads((ART/'eda/report.json').read_text()),indent=2,ensure_ascii=False))


## Optional exact baseline before ML
If test files are available, set RUN_TEST=True above. This baseline needs no trained model or retrieval indexes. Upload its matching_results.tsv to the challenge portal yourself. Without test files, this step is skipped; no leaderboard score is claimed.


In [ ]:
if RUN_TEST:
    run('src/prepare_data.py','--input',DRIVE_ROOT/'input/test','--output',DATA/'test','--split','test')
    phase(['src/data_integrity.py','--data',DATA/'test','--split','test','--output',ART/'test_data_integrity.json'],
          ['src/io_utils.py','--data',DATA/'test','--split','test','--index',TEST_INDEX,'--config',CONFIG],
          ['src/exact_baseline.py','--index',TEST_INDEX,'--output',EXACT_OUTPUT])
    print('Exact baseline:', SAVED/'output_exact/matching_results.tsv')
else:
    print('No test files requested; exact leaderboard baseline deferred.')


## Normalize and build fresh training indexes
Raw Unicode and combining marks remain intact. Optional transliteration creates additional fields. Target vocabulary is sampled; every target row is indexed and searched. Completed TF-IDF shards resume; incomplete catalog ingestion restarts.

In [ ]:
phase(['src/io_utils.py','--data',DATA/'train','--split','train','--index',INDEX,'--config',CONFIG])
phase(['src/tfidf_retriever.py','--index',INDEX,'--config',CONFIG])
phase(['src/bm25_retriever.py','--index',INDEX,'--config',CONFIG])


## Compare retrieval channels and inspect similarity distributions
Comparisons use training groups only. They never inject missed positives. The report compares the global K=20 and K=50 budgets, records candidate counts and retrieval time, and recommends targeted rescue if the configured failure floor is missed.

In [ ]:
phase(['src/retrieval_evaluation.py','--index',INDEX,'--config',CONFIG,'--output',ART/'retrieval_comparison'],
      ['src/eda.py','--data',DATA/'train','--index',INDEX,'--config',CONFIG,'--pairs-only','--output',ART/'eda'])
print('Retrieval:',ART/'retrieval_comparison/metrics.json')
print('Transliteration comparison:',ART/'retrieval_comparison/transliteration.json')


## Train both models, calibrate and evaluate
The approximately 60/20/20 grouped split is reproducible. LightGBM is the preselected primary model; the holdout does not select a model or threshold. Training restarts from scratch when rerun, while completed feature shards are reused.

In [ ]:
pilot = json.loads((ART/'retrieval_comparison/metrics.json').read_text())
selected = pilot['global_cap_pilot'][f"k_{cfg['blocking']['final_k']}"]['candidate_micro_recall']
if selected is None or selected < cfg['blocking']['failure_recall_floor']:
    raise RuntimeError('Selected K has insufficient measured recall. Review the pilot before training; do not scale a weak blocker.')
phase(['src/train.py','--index',INDEX,'--config',CONFIG,'--work',WORK,'--final',FINAL,
       '--retrieval-output',ART/'retrieval_comparison','--error-output',ART/'error_analysis','--skip-retrieval-eval'])
report = json.loads((WORK/'metrics.json').read_text())
print(json.dumps({name:result['holdout']['all'] for name,result in report['models'].items()},indent=2))
print('Model:',FINAL/'model.joblib')
print('Threshold:',FINAL/'threshold.json')
(ART/'source_commit.txt').write_text(subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True))
with (ART/'environment.txt').open('w') as f:
    subprocess.run([PY,'-m','pip','freeze'],stdout=f,check=True)
checkpoint()


## Test prediction — disabled until test files arrive
Set `RUN_TEST=True` in the config cell. No test labels are read. The integrity check and exact-match baseline run before the expensive hybrid retrieval. New test-target indexes use the settings saved with the trained model.

In [ ]:
if RUN_TEST:
    names = [f'test_source{i}.tsv' for i in (1,2,3)]
    if not all((DATA/'test'/name).exists() for name in names):
        run('src/prepare_data.py','--input',DRIVE_ROOT/'input/test','--output',DATA/'test','--split','test')
    phase(['src/data_integrity.py','--data',DATA/'test','--split','test','--output',ART/'test_data_integrity.json'],
          ['src/io_utils.py','--data',DATA/'test','--split','test','--index',TEST_INDEX,'--config',FINAL/'model_config.json'],
          ['src/exact_baseline.py','--index',TEST_INDEX,'--output',EXACT_OUTPUT],
          ['src/inference.py','--data',DATA/'test','--index',TEST_INDEX,'--model-dir',FINAL,'--output',OUTPUT],
          ['src/validate_submission.py','--index',TEST_INDEX,'--output',OUTPUT])
    print('Exact baseline:',EXACT_OUTPUT/'matching_results.tsv')
    print('Hybrid predictions:',OUTPUT/'matching_results.tsv')
else:
    print('Test prediction skipped; no test ground truth is needed.')


## Final package
Complete the team and actual results in `Documentation_template.md` with Colab’s file editor, then run this cell. It creates a ZIP but does not upload to the challenge portal.

In [ ]:
if RUN_TEST:
    if TEAM == 'YOUR_TEAM':raise ValueError('Set TEAM and complete Documentation_template.md before packaging.')
    phase(['src/package_submission.py','--team',TEAM,'--hybrid-index',TEST_INDEX,'--model-dir',FINAL,'--output',OUTPUT])
    print('Leaderboard TSV:',SAVED/'output_hybrid/matching_results.tsv')
    print('Final package:',SAVED/'output_hybrid/submission.zip')
else:
    print('Packaging waits for test predictions.')
